# Likelihood Optimization of gas Kinematics in IFUs (LOKI)
## Data manipulation example: Extracting a sub-cube from a full data cube

Michael Reefe

This example notebook provides a quick tutorial on how to use LOKI to extract a sub-cube from a full data cube.  This could be useful if you want to study one region of your cube in more detail.  The sub-cube can be used for a Loki analysis in the same way as a full cube.

First things first, we need to import the LOKI code. Remember we need to activate our project first (refer to the installation section of the README). We can do so simply by:

In [1]:
using Pkg
Pkg.activate(dirname(@__DIR__))
Pkg.instantiate()
Pkg.precompile()

using Loki

  Activating project at `~/Dropbox/Astrophysics/Codes/Loki`


WebIO._IJuliaInit()

Now we want to load in our data. For this example, we'll be using the data for the Sculptor Galaxy, which is located in the same folder as this notebook. 

We will first load in one observation, then correct it in the same way you would for a typical Loki analysis.

In [2]:
# The redshift of the target object: NGC 253 (Sculptor)
z = 0.000807

# The semicolon at the end suppresses printing the output Observation object, which is long and not very enlightening
obs = from_fits(["jw01701-c1002_t003_miri_ch3-medium_s3d.fits.gz"], z);

# This puts the observation in the rest frame, masks out bad spaxels, dereddens it, and logarithmically rebins it in wavelength
correct!(obs)

[ Info: Initializing DataCube struct from jw01701-c1002_t003_miri_ch3-medium_s3d.fits.gz
[ Info: Using SFD98 dust map at (α=11.888509583333333°, δ=-25.287788888888883°): E(B-V)=0.018742157430445203


Observation(Dict{Any, DataCube}(:B3 => DataCube{Vector{Unitful.Quantity{Float64, 𝐋, Unitful.FreeUnits{(μm,), 𝐋, nothing}}}, Array{Unitful.Quantity{Float64, 𝐌 𝐓⁻², Unitful.FreeUnits{(erg, Hz⁻¹, cm⁻², s⁻¹, sr⁻¹), 𝐌 𝐓⁻², nothing}}, 3}}(Unitful.Quantity{Float64, 𝐋, Unitful.FreeUnits{(μm,), 𝐋, nothing}}[13.33049244515671 μm, 13.332798160287632 μm, 13.335104274227696 μm, 13.337410787045883 μm, 13.339717698811187 μm, 13.342025009592609 μm, 13.344332719459166 μm, 13.346640828479886 μm, 13.348949336723809 μm, 13.351258244259988 μm  …  15.529934121972241 μm, 15.532620264606098 μm, 15.535306871849928 μm, 15.53799394378409 μm, 15.54068148048896 μm, 15.543369482044927 μm, 15.546057948532388 μm, 15.548746880031775 μm, 15.55143627662351 μm, 15.55412613838804 μm], Unitful.Quantity{Float64, 𝐌 𝐓⁻², Unitful.FreeUnits{(erg, Hz⁻¹, cm⁻², s⁻¹, sr⁻¹), 𝐌 𝐓⁻², nothing}}[NaN erg Hz⁻¹ cm⁻² s⁻¹ sr⁻¹ NaN erg Hz⁻¹ cm⁻² s⁻¹ sr⁻¹ … NaN erg Hz⁻¹ cm⁻² s⁻¹ sr⁻¹ NaN erg Hz⁻¹ cm⁻² s⁻¹ sr⁻¹; NaN erg Hz⁻¹ cm⁻² s⁻¹ sr⁻¹ NaN e

Now we will extract a sub-cube using the `make_subcube` function.  All we have to do is specify the minimum and maximum x and y pixel coordinates to extract the cube from.  The output will be an entirely new object, leaving the original unmodified.

For this example, we'll extract a small 6x6 cube.

The arguments, in order, are xmin, xmax, ymin, ymax.
Limits are inclusive and 1-based.  

In [3]:
obs_small = make_subcube(obs, 22, 27, 22, 27)

[ Info: Making a sub-cube with slices (22:27,22:27,1:893)


Observation(Dict{Any, DataCube}(:B3 => DataCube{Vector{Unitful.Quantity{Float64, 𝐋, Unitful.FreeUnits{(μm,), 𝐋, nothing}}}, Array{Unitful.Quantity{Float64, 𝐌 𝐓⁻², Unitful.FreeUnits{(erg, Hz⁻¹, cm⁻², s⁻¹, sr⁻¹), 𝐌 𝐓⁻², nothing}}, 3}}(Unitful.Quantity{Float64, 𝐋, Unitful.FreeUnits{(μm,), 𝐋, nothing}}[13.33049244515671 μm, 13.332798160287632 μm, 13.335104274227696 μm, 13.337410787045883 μm, 13.339717698811187 μm, 13.342025009592609 μm, 13.344332719459166 μm, 13.346640828479886 μm, 13.348949336723809 μm, 13.351258244259988 μm  …  15.529934121972241 μm, 15.532620264606098 μm, 15.535306871849928 μm, 15.53799394378409 μm, 15.54068148048896 μm, 15.543369482044927 μm, 15.546057948532388 μm, 15.548746880031775 μm, 15.55143627662351 μm, 15.55412613838804 μm], Unitful.Quantity{Float64, 𝐌 𝐓⁻², Unitful.FreeUnits{(erg, Hz⁻¹, cm⁻², s⁻¹, sr⁻¹), 𝐌 𝐓⁻², nothing}}[1.0211698612344836e-13 erg Hz⁻¹ cm⁻² s⁻¹ sr⁻¹ 9.232659595269117e-14 erg Hz⁻¹ cm⁻² s⁻¹ sr⁻¹ … 7.066151619549274e-14 erg Hz⁻¹ cm⁻² s⁻¹ sr⁻¹ 6.810

Our last pre-processing checklist items are to:
   1. Interpolate any NaNs in the data -- this is only done for spaxels that are less than 50% NaNs, the others are treated as a lost cause.
   2. Replace the pipeline-produced errors with statistically calculated errors (from the std.dev. of the residuals of a cubic spline spline)

You can do this either before or after extracting your subcube.  But if you do it after (as we've done here), it will not affect the full cube, which is not a separate object from the sub-cube.

In [4]:
channel = :B3

# We interpolate any rogue NaNs using a linear interpolation, since the MPFIT minimizer does not handle NaNs well.
interpolate_nans!(obs_small.channels[channel])

# Finally, we calculate the statistical errors (i.e. the standard deviation of the residuals with a cubic spline fit)
# and replace the errors in the cube with these, since the provided errors are typically underestimated.
# You can skip this step if you wish to use the default errors.
calculate_statistical_errors!(obs_small.channels[channel])

# Save the pre-processed data as a FITS file so it can be quickly reloaded later
save_fits(".", obs_small, [channel]);

[ Info: Interpolating NaNs in cube with channel 3, band MEDIUM:
[ Info: Calculating statistical errors for each spaxel...


Progress: 100%|█████████████████████████████████████████| Time: 0:00:00


[ Info: Writing FITS file from Observation object


Now, I'm not actually going to attempt to fit this cube, because it's not the point of this example, but I will show you what the subcube looks like with some of the visualization helper functions.

In [5]:
using Cosmology
cosmo = cosmology(;h=0.7, OmegaM=0.27, OmegaK=0., OmegaR=1e-5)

# The "plot_2d" function plots a 2D projection of the cube -- by default, this does an integration along the wavelength axis,
# but if the "slice" argument is specified, it can also show a single wavelength slice at the given coordinate.
# "intensity" and "err" booleans enable/disable plotting the intensity and the error
# "logᵢ" and "logₑ" if set to an integer  will make the colorbar logarithmic with that integer's base
# "colormap" changes the colormap (default: cubehelix) (you need to import python's matplotlib colormap package to use this)
# "name", if given, adds a plot title
# "z" is the redshift and must be provided to add a physical scale bar annotation to the plot, otherwise only an angular scale is added
# "cosmo" specifies the cosmology and it also must be provided along with "z" to add the physical scale bar annotation
# "aperture", if given, plots an aperture on top of the 2D map; use the make_aperture function to create an aperture
# "disable_psfcirc", if true, turns off the PSF circle in the plot, which is useful for small subcubes because it can become 
#     very large relative to the size of the cube.
plot_2d(obs.channels[channel], "Sculptor_fullcube_2d.pdf"; intensity=true, err=true, logᵢ=10, logₑ=nothing, 
        name=nothing, slice=nothing, z=obs.z, cosmo=cosmo, aperture=nothing)
plot_2d(obs_small.channels[channel], "Sculptor_subcube_2d.pdf"; intensity=true, err=true, logᵢ=10, logₑ=nothing, 
        name=nothing, slice=nothing, z=obs_small.z, cosmo=cosmo, aperture=nothing, disable_psfcirc=true)

# The "plot_1d" function plots a 1D projection of the cube -- by default, this does an integration along the 2 spatial axes,
# but if the "spaxel" argument is specified, it can also show a single spaxel's spectrum at the given coordinates.
# "intensity", "err", "name", and "logᵢ" work the same way as they do in "plot_2d", but there is no "logₑ" argument 
#       because logᵢ now affects both the intensity and the errors, which are placed on the same axes
# "linestyle" works like the matplotlib argument of the same name
plot_1d(obs.channels[channel], "Sculptor_fullcube_1d.pdf"; intensity=true, err=true, logᵢ=10, spaxel=(25,25), linestyle="-")
plot_1d(obs_small.channels[channel], "Sculptor_subcube_1d.pdf"; intensity=true, err=true, logᵢ=10, spaxel=(4,4), linestyle="-")

Here are the output 2D plots, showing the intensity on the left and the error on the right.

![](./Sculptor_fullcube_2d.png)
![](./Sculptor_subcube_2d.png)

And now the 1D plots.  These are both showing the same spaxel in each cube, so they are identical.

![](./Sculptor_fullcube_1d.png)
![](./Sculptor_subcube_1d.png)
